# 06c · `aede` en grilla de anchos, pesos y maxcl (iteración 3b, tareas D4 y D5)

**Objetivo.** Correr la EPCA del asesor (`aede`, port de Python `src/camiz_port.py`, D6) sobre la
grilla declarada de antemano en D10 (`decisiones.qmd`) y en el plan de la iteración 3b
(`asesoria/2026-10-21_plan-iteracion-3b.md`, tareas D4 y D5), en las dos vías (D1 y D2):

- anchos $n_w$ ∈ {24, 36, 48, 60} meses, paso de un mes, tres dimensiones;
- pesos uniformes y gaussianos (`nsd = 3`);
- Fisher sobre λ1 con maxcl ∈ {10, 15, 20};
- criterios: codo por caída máxima y Calinski-Harabász con $k$ y con $k-1$ grados de libertad
  (D11: el CH se informa y no selecciona).

La grilla se lee como **multiescala**, no como búsqueda de la ventana «correcta» (D10), y cada
$k^*$ se leerá contra el nulo de estructura constante (notebook 07b), no contra los episodios.

**Paridad (D6).** La referencia es la versión en R (`R/camiz_port.R`, Rmd, bloque
`b3b-aede-grilla`), que escribe `R_aede_cortes.csv` en formato largo. Si existe, este notebook
lo compara con el suyo ($k$ y fechas de corte exactos) y falla si difieren; la tabla de paridad
del Rmd hace la comparación inversa. Las filas con `nw = 60` y `maxcl = 10` son las de la
iteración 3a.

**Convención de fechado.** La fecha de cada ventana es la de su fila central, $i + n_w/2$
(1-based), como `rownames(serie)[(1 + nw/2):(wn + nw/2)]` en `aede_r`.

**Semillas.** `aede` y Fisher son determinísticos. El único sorteo es el del umbral de Horn de
la sección 3 (D5): `SEMILLA = 2026`, con el generador PCG64 de numpy (distinto del de R; ver
la excepción de generadores de la iteración 3a).

**Salidas** (niveles en `data/results/niveles/`, transformadas en `data/results/`):
`06c_aede_cortes.csv` y, en la sección 3, `06c_aede_valores_propios.csv`,
`06c_aede_valores_propios_resumen.csv` y `06c_aede_cruces.csv`.

In [1]:
# ---- Configuración --------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src import camiz_port as cp
from src.datos import cargar_vias

SEMILLA = 2026
NW_GRILLA = (24, 36, 48, 60)
PESOS = {'uniformes': 0, 'gaussianos_nsd3': 3}
MAXCL_GRILLA = (10, 15, 20)
NDIM = 3
CRITERIOS = [('codo (caída máxima)', 'k_codo', 'cortes_codo'),
             ('Calinski-Harabász k gl (aede)', 'k_ch', 'cortes_ch'),
             ('Calinski-Harabász k-1 gl (habitual)', 'k_ch1', 'cortes_ch1')]
DIR_T = RAIZ / 'data' / 'results'
DIR_N = DIR_T / 'niveles'

X, Xt = cargar_vias(RAIZ)
VIAS = {'niveles': (X, DIR_N), 'transformadas': (Xt, DIR_T)}
for via, (M, _) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1])

niveles (228, 17) 2007-01 → 2025-12
transformadas (227, 17) 2007-02 → 2025-12


## 1. Grilla de `aede` (D4)

In [2]:
corridas = {}      # (vía, nw, pesos) -> salida de aede con maxcl = 10 (para la sección 3)
cortes = {}
for via, (M, d) in VIAS.items():
    filas = []
    for nw in NW_GRILLA:
        for pesos, nsd in PESOS.items():
            for maxcl in MAXCL_GRILLA:
                a = cp.aede(M, nw=nw, ndim=NDIM, nsd=nsd, maxcl=maxcl)
                r = cp.resumen_aede(a)
                if maxcl == MAXCL_GRILLA[0]:
                    corridas[(via, nw, pesos)] = a
                for crit, kk, cc in CRITERIOS:
                    filas.append({'nw': nw, 'maxcl': maxcl, 'pesos': pesos, 'criterio': crit, 'k': r[kk],
                                  'k_en_tope': r[kk] == maxcl, 'cortes': ' '.join(r[cc])})
    tab = pd.DataFrame(filas)
    tab.to_csv(d / '06c_aede_cortes.csv', index=False)
    cortes[via] = tab
    print(via, len(tab), 'filas')

niveles 72 filas


transformadas 72 filas


### 1.1 Comparación con R y no regresión

Con `nw = 60` y `maxcl = 10`, las filas deben ser las del `999_port_aede_cortes.csv` de la
iteración 3a (que reproducía el `R_aede_cortes.csv` de entonces).

In [3]:
for via, (M, d) in VIAS.items():
    py = pd.read_csv(d / '06c_aede_cortes.csv', keep_default_na=False)
    viejo = pd.read_csv(d / '999_port_aede_cortes.csv', keep_default_na=False)
    s60 = py[(py.nw == 60) & (py.maxcl == 10)].reset_index(drop=True)
    assert (s60[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()
            == viejo[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()).all(), via
    f_r = d / 'R_aede_cortes.csv'
    ref = pd.read_csv(f_r, keep_default_na=False)
    if 'nw' in ref.columns:
        clave = lambda t: [f'{a} {b} {c} {e} {k} {str(t_).upper()} {ct}' for a, b, c, e, k, t_, ct in
                           zip(t.nw, t.maxcl, t.pesos, t.criterio, t.k, t.k_en_tope, t.cortes)]
        n_dif = sum(a != b for a, b in zip(clave(ref), clave(py)))
        assert len(ref) == len(py) and n_dif == 0, (via, n_dif)
        print(f'{via}: coincide con {f_r.name} ({len(py)} filas) y reproduce las filas de 3a')
    else:
        print(f'{via}: {f_r.name} aún en el formato de 3a (ejecutar el Rmd); reproduce las filas de 3a')

niveles: coincide con R_aede_cortes.csv (72 filas) y reproduce las filas de 3a
transformadas: coincide con R_aede_cortes.csv (72 filas) y reproduce las filas de 3a


In [4]:
# Vista compacta: k por criterio y maxcl (* = en el tope)
for via in VIAS:
    t = cortes[via].copy()
    t['celda'] = t.k.astype(str) + np.where(t.k_en_tope, '*', '')
    print(via)
    display(t.pivot_table(index=['nw', 'pesos', 'criterio'], columns='maxcl', values='celda', aggfunc='first'))

niveles


maxcl                                                    10   15   20
nw pesos           criterio                                          
24 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    6    6    6
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    6  15*  20*
                   codo (caída máxima)                    6    6    6
36 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2  15*  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
48 gaussianos_nsd3 Calinski-Harabász k gl (aede)          3  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
60 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*   14  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2

transformadas


maxcl                                                    10   15   20
nw pesos           criterio                                          
24 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*   14  20*
                   Calinski-Harabász k-1 gl (habitual)  10*   14  20*
                   codo (caída máxima)                    4   10   10
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4
36 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    3    3    3
   uniformes       Calinski-Harabász k gl (aede)          9  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    9    9  20*
                   codo (caída máxima)                    3    3    3
48 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    3    3    3
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    7  15*  20*
                   codo (caída máxima)                    4    4    4
60 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4